# Extract Shot Lengths

This notebook detects shot boundaries for all real `.mp4` files in `/Volumes/Extreme SSD/Video/Chinese1949-66`, skips macOS `._*.mp4` files, and writes resume-safe CSV outputs after each completed movie.

Resume is movie-level: if the kernel, Jupyter, or computer stops during one movie, that movie is recomputed next time, while already completed movies are skipped if their file size and mtime still match.

In [11]:
# Optional dependency installer/repair cell.
# Keep AUTO_INSTALL_DEPENDENCIES=True if this is the first run or cv2/scenedetect import fails.
AUTO_INSTALL_DEPENDENCIES = True

import importlib
import subprocess
import sys

def _missing_or_broken_modules(module_names):
    broken = []
    for module_name in module_names:
        try:
            importlib.import_module(module_name)
        except Exception as exc:
            broken.append((module_name, repr(exc)))
    return broken

required_modules = ["pandas", "tqdm", "scenedetect", "cv2"]
broken_modules = _missing_or_broken_modules(required_modules)

if broken_modules and AUTO_INSTALL_DEPENDENCIES:
    print("Installing/repairing dependencies because these imports failed:")
    for name, err in broken_modules:
        print(f"  - {name}: {err}")
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-U",
        "scenedetect[opencv]",
        "opencv-python-headless",
        "pandas",
        "tqdm",
    ])
    broken_modules = _missing_or_broken_modules(required_modules)

if broken_modules:
    details = "\n".join(f"{name}: {err}" for name, err in broken_modules)
    raise RuntimeError(
        "Some dependencies are still missing or broken. "
        "Try restarting the Jupyter kernel and rerunning this cell.\n" + details
    )

print("Dependencies look ready.")

Dependencies look ready.


In [12]:
from __future__ import annotations

import json
import math
import os
import subprocess
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from scenedetect import SceneManager, open_video
from scenedetect.detectors import ContentDetector
from tqdm.auto import tqdm

VIDEO_DIR = Path("/Volumes/Extreme SSD/Video/Chinese1949-66")
OUTPUT_DIR = Path("./shot_length_results")

THRESHOLD = 27.0
MIN_SCENE_LEN_FRAMES = 12
ROUND_SECONDS = 3

# Use a small number for smoke testing, e.g. MAX_MOVIES = 2. Use None for all movies.
MAX_MOVIES = None

PROGRESS_CSV = OUTPUT_DIR / "movie_progress.csv"
LONG_CSV = OUTPUT_DIR / "shot_lengths_long.csv"
WIDE_CSV = OUTPUT_DIR / "shot_lengths_wide.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Video directory: {VIDEO_DIR}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

Video directory: /Volumes/Extreme SSD/Video/Chinese1949-66
Output directory: /Users/caoshuangyuan/Desktop/Workspace/DirtyPilot/shot_length_results


In [13]:
def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def list_video_files(video_dir: Path) -> list[Path]:
    if not video_dir.exists():
        raise FileNotFoundError(f"Video directory does not exist: {video_dir}")
    return sorted(
        path
        for path in video_dir.iterdir()
        if path.is_file()
        and path.suffix.lower() == ".mp4"
        and not path.name.startswith("._")
    )


def movie_metadata(video_path: Path) -> dict:
    stat = video_path.stat()
    return {
        "film": video_path.stem,
        "video_path": str(video_path),
        "file_size": int(stat.st_size),
        "mtime": int(stat.st_mtime_ns),
    }


def read_csv_or_empty(path: Path, columns: list[str]) -> pd.DataFrame:
    if path.exists() and path.stat().st_size > 0:
        return pd.read_csv(path)
    return pd.DataFrame(columns=columns)


def atomic_write_csv(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp_path, index=False, encoding="utf-8-sig")
    os.replace(tmp_path, path)


def load_progress() -> pd.DataFrame:
    columns = [
        "film",
        "status",
        "video_path",
        "file_size",
        "mtime",
        "shot_count",
        "error",
        "started_at",
        "finished_at",
        "detector",
        "threshold",
        "min_scene_len_frames",
    ]
    return read_csv_or_empty(PROGRESS_CSV, columns)


def load_long_results() -> pd.DataFrame:
    columns = ["film", "shot_index", "duration_sec", "start_sec", "end_sec", "video_path"]
    return read_csv_or_empty(LONG_CSV, columns)


def replace_rows_by_film(df: pd.DataFrame, film: str, replacement: pd.DataFrame) -> pd.DataFrame:
    if not df.empty and "film" in df.columns:
        df = df[df["film"] != film].copy()
    return pd.concat([df, replacement], ignore_index=True)


def upsert_progress(progress_df: pd.DataFrame, row: dict) -> pd.DataFrame:
    replacement = pd.DataFrame([row])
    return replace_rows_by_film(progress_df, row["film"], replacement)


def write_all_outputs(progress_df: pd.DataFrame, long_df: pd.DataFrame) -> None:
    progress_df = progress_df.sort_values("film", kind="stable").reset_index(drop=True)
    long_df = long_df.sort_values(["film", "shot_index"], kind="stable").reset_index(drop=True)
    atomic_write_csv(progress_df, PROGRESS_CSV)
    atomic_write_csv(long_df, LONG_CSV)
    atomic_write_csv(make_wide_results(long_df), WIDE_CSV)


def make_wide_results(long_df: pd.DataFrame) -> pd.DataFrame:
    if long_df.empty:
        return pd.DataFrame(columns=["film"])
    wide_rows = []
    max_shots = int(long_df["shot_index"].max())
    width = max(3, len(str(max_shots)))
    for film, group in long_df.sort_values(["film", "shot_index"]).groupby("film", sort=True):
        row = {"film": film}
        for _, shot in group.iterrows():
            shot_index = int(shot["shot_index"])
            row[f"shot_{shot_index:0{width}d}"] = shot["duration_sec"]
        wide_rows.append(row)
    return pd.DataFrame(wide_rows)


def is_complete_and_unchanged(progress_df: pd.DataFrame, metadata: dict) -> bool:
    if progress_df.empty or "film" not in progress_df.columns:
        return False
    rows = progress_df[progress_df["film"] == metadata["film"]]
    if rows.empty:
        return False
    row = rows.iloc[-1]
    if str(row.get("status", "")) != "complete":
        return False
    try:
        return (
            int(row.get("file_size", -1)) == metadata["file_size"]
            and int(row.get("mtime", -1)) == metadata["mtime"]
        )
    except Exception:
        return False

In [14]:
def ffprobe_duration_seconds(video_path: Path) -> float:
    cmd = [
        "ffprobe",
        "-v",
        "error",
        "-show_entries",
        "format=duration",
        "-of",
        "json",
        str(video_path),
    ]
    completed = subprocess.run(cmd, check=True, capture_output=True, text=True)
    payload = json.loads(completed.stdout)
    return float(payload["format"]["duration"])


def detect_shot_lengths(video_path: Path) -> pd.DataFrame:
    video = open_video(str(video_path))
    scene_manager = SceneManager()
    scene_manager.add_detector(
        ContentDetector(threshold=THRESHOLD, min_scene_len=MIN_SCENE_LEN_FRAMES)
    )
    scene_manager.detect_scenes(video=video, show_progress=True)
    scene_list = scene_manager.get_scene_list()

    if not scene_list:
        total_duration = ffprobe_duration_seconds(video_path)
        scene_list = [(None, None)]
    else:
        total_duration = None

    rows = []
    for index, scene in enumerate(scene_list, start=1):
        if scene == (None, None):
            start_sec = 0.0
            end_sec = total_duration
        else:
            start_time, end_time = scene
            start_sec = float(start_time.get_seconds())
            end_sec = float(end_time.get_seconds())

        duration_sec = max(0.0, end_sec - start_sec)
        rows.append({
            "film": video_path.stem,
            "shot_index": index,
            "duration_sec": round(duration_sec, ROUND_SECONDS),
            "start_sec": round(start_sec, ROUND_SECONDS),
            "end_sec": round(end_sec, ROUND_SECONDS),
            "video_path": str(video_path),
        })

    return pd.DataFrame(rows, columns=[
        "film",
        "shot_index",
        "duration_sec",
        "start_sec",
        "end_sec",
        "video_path",
    ])

In [15]:
def process_movies(max_movies: int | None = MAX_MOVIES) -> tuple[pd.DataFrame, pd.DataFrame]:
    videos = list_video_files(VIDEO_DIR)
    progress_df = load_progress()
    long_df = load_long_results()

    candidates = []
    skipped = 0
    for video_path in videos:
        metadata = movie_metadata(video_path)
        if is_complete_and_unchanged(progress_df, metadata):
            skipped += 1
            continue
        candidates.append((video_path, metadata))

    if max_movies is not None:
        candidates = candidates[:max_movies]

    print(f"Found {len(videos)} real .mp4 files; skipped {skipped} already complete unchanged movies.")
    print(f"Processing {len(candidates)} movie(s) this run. MAX_MOVIES={max_movies}")

    for video_path, metadata in tqdm(candidates, desc="Movies"):
        film = metadata["film"]
        started_at = utc_now_iso()
        processing_row = {
            **metadata,
            "status": "processing",
            "shot_count": 0,
            "error": "",
            "started_at": started_at,
            "finished_at": "",
            "detector": "PySceneDetect ContentDetector",
            "threshold": THRESHOLD,
            "min_scene_len_frames": MIN_SCENE_LEN_FRAMES,
        }
        progress_df = upsert_progress(progress_df, processing_row)
        write_all_outputs(progress_df, long_df)

        try:
            movie_long_df = detect_shot_lengths(video_path)
            long_df = replace_rows_by_film(long_df, film, movie_long_df)
            complete_row = {
                **metadata,
                "status": "complete",
                "shot_count": int(len(movie_long_df)),
                "error": "",
                "started_at": started_at,
                "finished_at": utc_now_iso(),
                "detector": "PySceneDetect ContentDetector",
                "threshold": THRESHOLD,
                "min_scene_len_frames": MIN_SCENE_LEN_FRAMES,
            }
            progress_df = upsert_progress(progress_df, complete_row)
        except Exception as exc:
            failed_row = {
                **metadata,
                "status": "failed",
                "shot_count": 0,
                "error": repr(exc),
                "started_at": started_at,
                "finished_at": utc_now_iso(),
                "detector": "PySceneDetect ContentDetector",
                "threshold": THRESHOLD,
                "min_scene_len_frames": MIN_SCENE_LEN_FRAMES,
            }
            progress_df = upsert_progress(progress_df, failed_row)
            print(f"FAILED: {film}: {exc!r}")

        write_all_outputs(progress_df, long_df)

    return progress_df, long_df


progress_df, long_df = process_movies(MAX_MOVIES)

Found 454 real .mp4 files; skipped 360 already complete unchanged movies.
Processing 94 movie(s) this run. MAX_MOVIES=None


Movies:   0%|          | 0/94 [00:00<?, ?it/s]



























































































































































































































































































































































































































































































































































































































































































































































































































































































































































































In [16]:
videos = list_video_files(VIDEO_DIR)
progress_df = load_progress()
long_df = load_long_results()
wide_df = make_wide_results(long_df)

complete_count = int((progress_df.get("status", pd.Series(dtype=str)) == "complete").sum())
failed_count = int((progress_df.get("status", pd.Series(dtype=str)) == "failed").sum())
processing_count = int((progress_df.get("status", pd.Series(dtype=str)) == "processing").sum())
remaining_count = max(0, len(videos) - complete_count)

print(f"Real .mp4 files: {len(videos)}")
print(f"Complete: {complete_count}")
print(f"Failed: {failed_count}")
print(f"Processing/incomplete from an interrupted run: {processing_count}")
print(f"Remaining not complete: {remaining_count}")
print(f"Progress CSV: {PROGRESS_CSV.resolve()}")
print(f"Long CSV: {LONG_CSV.resolve()}")
print(f"Wide CSV: {WIDE_CSV.resolve()}")

display(progress_df.tail(10))
display(long_df.head(10))
display(wide_df.head(10))

Real .mp4 files: 454
Complete: 454
Failed: 0
Processing/incomplete from an interrupted run: 0
Remaining not complete: 0
Progress CSV: /Users/caoshuangyuan/Desktop/Workspace/DirtyPilot/shot_length_results/movie_progress.csv
Long CSV: /Users/caoshuangyuan/Desktop/Workspace/DirtyPilot/shot_length_results/shot_lengths_long.csv
Wide CSV: /Users/caoshuangyuan/Desktop/Workspace/DirtyPilot/shot_length_results/shot_lengths_wide.csv


,film,status,video_path,file_size,mtime,shot_count,error,started_at,finished_at,detector,threshold,min_scene_len_frames
444,魔术师的奇遇,complete,/Volumes/Extreme SSD/Video/Chinese1949-66/魔术师的...,209501023,1681064754000000000,193,NaN,2026-07-08T21:58:17+00:00,2026-07-08T21:58:58+00:00,PySceneDetect ContentDetector,27.0,12
445,鲁班的传说,complete,/Volumes/Extreme SSD/Video/Chinese1949-66/鲁班的传...,343096947,1680908528000000000,51,NaN,2026-07-08T21:59:02+00:00,2026-07-08T22:00:35+00:00,PySceneDetect ContentDetector,27.0,12
446,鸡毛信,complete,/Volumes/Extreme SSD/Video/Chinese1949-66/鸡毛信.mp4,326772267,1680891016000000000,333,NaN,2026-07-08T22:00:39+00:00,2026-07-08T22:02:09+00:00,PySceneDetect ContentDetector,27.0,12
447,黄沙绿浪,complete,/Volumes/Extreme SSD/Video/Chinese1949-66/黄沙绿浪...,463625671,1681065904000000000,771,NaN,2026-07-08T22:02:12+00:00,2026-07-08T22:03:58+00:00,PySceneDetect ContentDetector,27.0,12
448,黄浦江的故事,complete,/Volumes/Extreme SSD/Video/Chinese1949-66/黄浦江的...,426626629,1681246116000000000,622,NaN,2026-07-08T22:04:01+00:00,2026-07-08T22:05:29+00:00,PySceneDetect ContentDetector,27.0,12
449,黎明的河边,complete,/Volumes/Extreme SSD/Video/Chinese1949-66/黎明的河...,298794943,1680894164000000000,319,NaN,2026-07-08T22:05:32+00:00,2026-07-08T22:07:03+00:00,PySceneDetect ContentDetector,27.0,12
450,黑山阻击战,complete,/Volumes/Extreme SSD/Video/Chinese1949-66/黑山阻击...,372069269,1680908914000000000,491,NaN,2026-07-08T22:07:07+00:00,2026-07-08T22:08:57+00:00,PySceneDetect ContentDetector,27.0,12
451,齐王求将,complete,/Volumes/Extreme SSD/Video/Chinese1949-66/齐王求将...,455989736,1681064072000000000,272,NaN,2026-07-08T22:09:00+00:00,2026-07-08T22:11:23+00:00,PySceneDetect ContentDetector,27.0,12
452,龙须沟,complete,/Volumes/Extreme SSD/Video/Chinese1949-66/龙须沟.mp4,473286170,1680890776000000000,1078,NaN,2026-07-08T22:11:27+00:00,2026-07-08T22:14:10+00:00,PySceneDetect ContentDetector,27.0,12
453,龙马精神,complete,/Volumes/Extreme SSD/Video/Chinese1949-66/龙马精神...,463910095,1681066110000000000,535,NaN,2026-07-08T22:14:14+00:00,2026-07-08T22:16:31+00:00,PySceneDetect ContentDetector,27.0,12


,film,shot_index,duration_sec,start_sec,end_sec,video_path
0,51号兵站,1,1.960,0.000,1.960,/Volumes/Extreme SSD/Video/Chinese1949-66/51号兵...
1,51号兵站,2,20.923,1.960,22.883,/Volumes/Extreme SSD/Video/Chinese1949-66/51号兵...
2,51号兵站,3,67.769,22.883,90.652,/Volumes/Extreme SSD/Video/Chinese1949-66/51号兵...
3,51号兵站,4,4.921,90.652,95.573,/Volumes/Extreme SSD/Video/Chinese1949-66/51号兵...
4,51号兵站,5,3.480,95.573,99.053,/Volumes/Extreme SSD/Video/Chinese1949-66/51号兵...
5,51号兵站,6,1.640,99.053,100.694,/Volumes/Extreme SSD/Video/Chinese1949-66/51号兵...
6,51号兵站,7,18.042,100.694,118.736,/Volumes/Extreme SSD/Video/Chinese1949-66/51号兵...
7,51号兵站,8,6.841,118.736,125.577,/Volumes/Extreme SSD/Video/Chinese1949-66/51号兵...
8,51号兵站,9,1.000,125.577,126.577,/Volumes/Extreme SSD/Video/Chinese1949-66/51号兵...
9,51号兵站,10,6.921,126.577,133.498,/Volumes/Extreme SSD/Video/Chinese1949-66/51号兵...


,film,shot_0001,shot_0002,shot_0003,shot_0004,shot_0005,shot_0006,shot_0007,shot_0008,shot_0009,...,shot_2780,shot_2781,shot_2782,shot_2783,shot_2784,shot_2785,shot_2786,shot_2787,shot_2788,shot_2789
0,51号兵站,1.960,20.923,67.769,4.921,3.480,1.640,18.042,6.841,1.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,一串项链,2.000,2.000,8.001,1.000,1.000,1.000,5.001,1.000,274.035,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,一件提案,174.080,24.600,4.240,147.200,40.000,243.760,30.360,38.120,8.880,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,一场风波,3985.960,442.360,530.280,29.360,13.120,14.680,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,一天一夜,56.808,42.606,11.722,21.963,1.480,3.120,30.404,3.200,12.242,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,一文钱,3.040,1.000,2.120,0.880,1.000,1.000,1.120,0.880,1.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,一日千里,1.720,0.960,3.841,0.960,1.920,0.960,90.012,4.841,2.960,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,一百个放心,2.000,1.000,3.000,1.000,1.000,1.000,12.002,1.000,12.002,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,一贯害人道,22.323,70.410,57.808,11.282,4.001,2.040,29.604,5.881,12.882,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,一颗红心,4.081,1.360,67.529,26.044,56.808,19.523,11.202,40.246,112.096,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Full Run

After the smoke test works, change `MAX_MOVIES = None` in the config cell and rerun the processing cell. The notebook will skip movies already marked complete with unchanged size/mtime.